# EasyRAG Observing RAG Artifacts

## What you will do

- create one tiny deterministic workspace with `EasyRAG`
- inspect the artifacts produced by `prepare -> insert -> query -> answer`
- compare pre-insert payloads with post-insert runtime metadata

## Why this matters

RAG systems often feel opaque because we only look at the final answer. This notebook keeps the intermediate artifacts visible. You will see what EasyRAG prepares before storage, what `QueryResult` exposes after retrieval, and what `get_stats()` summarizes at the workspace level.


## Source code anchors

- `easyrag.rag.orchestrator.EasyRAG.initialize_storages`
- `easyrag.rag.orchestrator.EasyRAG.ainsert`
- `easyrag.rag.orchestrator.EasyRAG.aquery`
- `easyrag.rag.orchestrator.EasyRAG.get_stats`
- `easyrag.rag.indexing.pipeline.build_insert_payloads`
- `easyrag.rag.types.QueryResult`


In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)

from easyrag.rag.indexing import build_insert_payloads


## Deterministic path

We start with the same kind of deterministic stub setup used in the tests. That keeps the notebook runnable without API keys and makes the artifact shapes stable across runs.


In [ ]:
artifact_tmp = tempfile.TemporaryDirectory()
artifact_rag = EasyRAG(
    working_dir=artifact_tmp.name,
    workspace="artifact-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
    reranker_func=_stub_reranker,
)
run_sync(artifact_rag.initialize_storages())

artifact_texts = [
    "# Architecture\nEasyRAG uses query rewrite and grounded retrieval for architecture guidance.\n## Retrieval\nHybrid retrieval uses rerank and chunk inspection.\n",
    "# Storage\nRetrieved evidence is packed before answer synthesis and storage review.\n## Policy\nAnswers stay citation-aware so readers can inspect the source chunks.\n",
]
artifact_ids = ["doc::architecture", "doc::storage"]
artifact_paths = ["docs/architecture.md", "docs/storage.md"]

prepared_documents = EasyRAG.prepare_documents(artifact_texts, ids=artifact_ids, file_paths=artifact_paths)
artifact_payloads = build_insert_payloads(artifact_rag, prepared_documents)

print("Prepared document metadata")
for document in prepared_documents:
    pprint(document.metadata)

print("\nPre-insert payload counts")
_print_json({key: len(value) if isinstance(value, list) else value for key, value in artifact_payloads.items() if key not in {"chunk_strategy_counts"}})
print("\nChunk strategy counts")
_print_json(artifact_payloads["chunk_strategy_counts"])


## Output inspection

The payload view is the easiest way to understand what the orchestrator will write. Notice that one logical document fans out into document records, summaries, chunks, vector records, graph nodes, graph edges, and document-status records.


In [ ]:
insert_stats = run_sync(artifact_rag.ainsert(artifact_texts, ids=artifact_ids, file_paths=artifact_paths))
artifact_result = run_sync(
    artifact_rag.aquery(
        "How does EasyRAG keep answers grounded?",
        QueryParam(mode="hybrid", enable_rerank=True, chunk_top_k=3),
    )
)
artifact_answer = run_sync(
    artifact_rag.aanswer(
        "How does EasyRAG keep answers grounded?",
        QueryParam(mode="hybrid", enable_rerank=True, chunk_top_k=3),
        AnswerParam(max_citations=2, max_context_chars=180),
    )
)
architecture_status = run_sync(artifact_rag.doc_status_storage.get_status("doc::architecture"))
workspace_tree = [str(path.relative_to(Path(artifact_tmp.name))) for path in sorted(Path(artifact_tmp.name).rglob("*"))]

print("Insert stats")
_print_json(insert_stats)
print("\nFirst document payload")
_print_json(artifact_payloads["documents"][0])
print("\nFirst chunk payload")
_print_json(artifact_payloads["chunks"][0])
print("\nQueryResult metadata")
_print_json(artifact_result.metadata)
print("\nFirst citation")
_print_json(artifact_result.citations[0])
print("\nFirst chunk metadata")
_print_json(artifact_result.chunks[0].metadata)
print("\nAnswer metadata")
_print_json(artifact_answer.metadata)
print("\nDocument status")
_print_json(architecture_status)
print("\nWorkspace tree")
pprint(workspace_tree)
print("\nWorkspace aggregate stats")
_print_json(run_sync(artifact_rag.get_stats()))


## Provider-backed path

The next cell uses the same teaching flow but swaps the deterministic retrieval helpers for the repo's default OpenAI-compatible providers when the environment is configured. The cell skips cleanly when credentials are missing.


In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_tmp = tempfile.TemporaryDirectory()
    provider_rag = EasyRAG(working_dir=provider_tmp.name, workspace="provider-artifact-demo")
    try:
        run_sync(provider_rag.initialize_storages())
        run_sync(provider_rag.ainsert(artifact_texts, ids=artifact_ids, file_paths=artifact_paths))
        provider_result = run_sync(
            provider_rag.aquery(
                "How does EasyRAG keep answers grounded?",
                QueryParam(mode="hybrid", chunk_top_k=3),
            )
        )
        print("Provider-backed metadata keys")
        pprint(sorted(provider_result.metadata.keys()))
        print("\nProvider-backed first citation")
        _print_json(provider_result.citations[0] if provider_result.citations else {})
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")
    finally:
        try:
            run_sync(provider_rag.finalize_storages())
        finally:
            provider_tmp.cleanup()


## What changed and why

The key change across the lifecycle is representation, not ownership. The same source text becomes prepared documents, then storage payloads, then ranked retrieval citations, then packed answer evidence. Keeping those intermediate objects visible makes debugging much easier than jumping straight from raw text to a final answer.


In [ ]:
run_sync(artifact_rag.finalize_storages())
artifact_tmp.cleanup()
print("Cleaned up the deterministic workspace.")


## Next steps

- Continue with [02_01_repo_loading_basics.ipynb](../02_data_loading/02_01_repo_loading_basics.ipynb) to see how repository-shaped inputs become canonical `Document` objects.
- Read [00-overview.md](../../docs/00-overview.md) for the full EasyRAG learning path.
- Keep [notebooks/README.md](../README.md) open as the stage-by-stage notebook map.
